# Assignment 1 — Q3: Indian Food Recipes Dataset — Acquisition, Wrangling and EDA
**Data Science Lab (CS2311)**

**Goal:** build a structured dataset of **200+ unique Indian dishes** covering all six regions (North, South, East, West, Central, North-East), with at least 10 attributes, then clean it, engineer features and answer analytical questions.

## Data sources and honest disclosure

No single website lists hundreds of Indian dishes with region, GI status, ingredients and cooking method all in one place, so we **combine sources** (which is also why the assignment asks us to remove duplicates "collected from multiple sources"):

| # | Source | What it gives |
|---|---|---|
| 1 | Wikipedia — *List of Indian dishes* (scraped) | ~300 dishes in regional tables: name, description (mentions main ingredients and cooking style), vegetarian/non-vegetarian, image URL, link to the dish's own article |
| 2 | Wikipedia — *List of geographical indications in India* (scraped) | Official GI register: product name, type ("Food stuff"), state, year. Gives **GI tag status** and extra GI-tagged foods |
| 3 | Small manual supplement (in this notebook) | ~20 well-known Central and North-East dishes, because these regions are thinly covered on source 1. Clearly labelled `source = 'Manual supplement'` |

**Attributes and where they come from:**

| Attribute | Origin |
|---|---|
| recipe name, description, veg/non-veg, image URL, source URL, region (table section) | scraped |
| state | found in the name/description ("from Assam", "Hyderabadi", "Bengali"...) or from the GI register |
| GI tag status | matched against the GI register |
| major ingredients | extracted from the description with an ingredient vocabulary |
| cooking method, category (breakfast/lunch/dinner/snack/dessert), recommended meal time | keyword rules on name + description (+ Wikipedia's "Dish Type" column where it exists) |
| preparation time, nutrition | **not published** in these sources → recorded as missing, then handled in cleaning |

**Region definition:** we standardise regions with the Government of India's **Zonal Councils** grouping of states (North, Central, East, West, South, North-East), so every state maps to exactly one region.

In [ ]:
import json
import os
import re
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

In [ ]:
DISHES_URL = 'https://en.wikipedia.org/wiki/List_of_Indian_dishes'
GI_URL = 'https://en.wikipedia.org/wiki/List_of_geographical_indications_in_India'
WIKI_ROOT = 'https://en.wikipedia.org'
HEADERS = {'User-Agent': 'CS2311-DataScienceLab/1.0 (student project; educational use)'}   # Wikipedia asks for a descriptive User-Agent
RAW_JSON = 'indian_recipes_raw.json'


def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.content, 'html.parser')

---
## (a) Scraping

### A robust table reader
Wikipedia tables sometimes use `rowspan` (one cell stretched over several rows, e.g. the same state for many GI products). A naive "one `<td>` per column" loop then shifts every following cell into the wrong column. The function below keeps a "carry-over" for stretched cells, so every row comes out with the right number of columns.

In [ ]:
def clean_cell(cell):
    for sup in cell.find_all('sup'):          # remove footnote markers like [12]
        sup.decompose()
    return cell.get_text(' ', strip=True)


def read_table(table):
    """Return (headers, rows) for an HTML table; each row is a list of the original <td>/<th> Tag objects."""
    trs = table.find_all('tr')
    headers = [clean_cell(th) for th in trs[0].find_all(['th', 'td'])]
    rows, carry = [], {}                      # carry: column index -> [cell, rows_left]
    for tr in trs[1:]:
        cells = tr.find_all(['td', 'th'], recursive=False)
        if not cells:
            continue
        row, col, it = [], 0, iter(cells)
        while col < len(headers):
            if col in carry:                  # a cell stretched down from a previous row
                cell, left = carry[col]
                row.append(cell)
                carry[col][1] -= 1
                if carry[col][1] == 0:
                    del carry[col]
                col += 1
                continue
            cell = next(it, None)
            if cell is None:
                break
            span = int(cell.get('rowspan', 1) or 1)
            for _ in range(int(cell.get('colspan', 1) or 1)):
                row.append(cell)
                if span > 1:
                    carry[col] = [cell, span - 1]
                col += 1
        if len(row) == len(headers):
            rows.append(row)
    return headers, rows

### Source 1: Wikipedia — List of Indian dishes
Each regional table sits under a heading (North, South, ...). For every table we look up the **nearest heading above it** to know the region. Columns: Name | Image | Description | Vegetarian/Non-Vegetarian (+ "Dish Type" in some tables).

In [ ]:
def region_from_heading(text):
    t = text.lower().replace(' ', '').replace('-', '')
    if 'northeast' in t: return 'North-East'
    if 'north' in t: return 'North'
    if 'south' in t: return 'South'
    if 'west' in t: return 'West'
    if 'east' in t: return 'East'
    if 'central' in t: return 'Central'
    return None


def find_col(headers, *keywords):
    for i, h in enumerate(headers):
        if any(k in h.lower() for k in keywords):
            return i
    return None


TABLE_ORDER = ['North-East', 'North', 'South', 'West', 'East']   # order of the regional tables on the page (backup)

def scrape_dishes():
    soup = get_soup(DISHES_URL)
    records, dish_table_no = [], -1
    for table in soup.find_all('table', class_='wikitable'):
        headers, rows = read_table(table)
        i_name, i_desc = find_col(headers, 'name'), find_col(headers, 'description')
        if i_name is None or i_desc is None:
            continue                                     # not a dish table
        dish_table_no += 1
        i_img, i_veg = find_col(headers, 'image'), find_col(headers, 'vegetarian')
        i_type = find_col(headers, 'type', 'course')
        heading = table.find_previous(['h2', 'h3', 'h4'])
        region = region_from_heading(heading.get_text()) if heading else None
        if region is None and dish_table_no < len(TABLE_ORDER):
            region = TABLE_ORDER[dish_table_no]          # backup: heading not recognised -> use table position
        for row in rows:
            name_cell = row[i_name]
            link = name_cell.find('a', href=True)
            img = row[i_img].find('img') if i_img is not None else None
            records.append({
                'recipe_name': clean_cell(name_cell),
                'description': clean_cell(row[i_desc]),
                'veg_nonveg': clean_cell(row[i_veg]) if i_veg is not None else None,
                'dish_type': clean_cell(row[i_type]) if i_type is not None else None,
                'region_section': region,
                'image_url': ('https:' + img['src']) if img and img['src'].startswith('//') else (img['src'] if img else None),
                'recipe_source_url': WIKI_ROOT + link['href'] if link and link['href'].startswith('/wiki/') else DISHES_URL,
                'source': 'Wikipedia: List of Indian dishes',
            })
    return records

### Source 2: Wikipedia — GI register (food stuff only)
The table has the columns *Geographical indication | Type | State/UT | Year*. We keep rows whose type is "Food stuff". These give us (1) the GI status of dishes and (2) additional GI-tagged foods.

In [ ]:
NOT_DISHES = ['feni', 'wine', 'liquor', 'beer', 'tea', 'coffee', 'water', 'salt', 'oil']

def scrape_gi_foods():
    soup = get_soup(GI_URL)
    foods = []
    for table in soup.find_all('table', class_='wikitable'):
        headers, rows = read_table(table)
        i_name = find_col(headers, 'indication', 'name')
        i_type, i_state = find_col(headers, 'type'), find_col(headers, 'state')
        if None in (i_name, i_type, i_state):
            continue
        i_year = find_col(headers, 'year')
        for row in rows:
            gi_type = clean_cell(row[i_type])
            name = clean_cell(row[i_name])
            if 'food' not in gi_type.lower() or any(w in name.lower() for w in NOT_DISHES):
                continue
            foods.append({'gi_name': name, 'gi_type': gi_type, 'state': clean_cell(row[i_state]),
                          'gi_year': clean_cell(row[i_year]) if i_year is not None else None})
    return foods

### Source 3: manual supplement (Central and North-East India)
Written from general knowledge to guarantee coverage of these two regions. Descriptions use the same style as Wikipedia (main ingredients + cooking style), so the same feature-extraction rules work on them.

In [ ]:
SUPPLEMENT = [
    # name, state, veg/non-veg, category hint, description
    ('Indori Poha', 'Madhya Pradesh', 'Vegetarian', 'Breakfast', 'Flattened rice steamed with onion, turmeric, mustard seeds, curry leaves and peanuts, topped with sev and lemon'),
    ('Bhutte ka Kees', 'Madhya Pradesh', 'Vegetarian', 'Snack', 'Grated corn cooked in milk with green chilli, mustard seeds and coriander'),
    ('Dal Bafla', 'Madhya Pradesh', 'Vegetarian', 'Lunch', 'Boiled and baked wheat flour dumplings dipped in ghee, served with spicy lentil dal'),
    ('Mawa Bati', 'Madhya Pradesh', 'Vegetarian', 'Dessert', 'Deep fried khoya dumplings stuffed with nuts and soaked in sugar syrup'),
    ('Bhopali Gosht Korma', 'Madhya Pradesh', 'Non-Vegetarian', 'Dinner', 'Mutton simmered in a curry of yogurt, onion, garlic, ginger and garam masala'),
    ('Chakki ki Shaak', 'Madhya Pradesh', 'Vegetarian', 'Lunch', 'Steamed wheat dough pieces cooked in a yogurt curry with spices'),
    ('Chila', 'Chhattisgarh', 'Vegetarian', 'Breakfast', 'Rice flour pancake roasted on a griddle, served with tomato chutney'),
    ('Fara', 'Chhattisgarh', 'Vegetarian', 'Snack', 'Steamed rice flour rolls tempered with mustard seeds, sesame and green chilli'),
    ('Aamat', 'Chhattisgarh', 'Vegetarian', 'Lunch', 'Mixed vegetable curry with bamboo shoot, lentils and spices cooked in a sour gravy'),
    ('Dehrori', 'Chhattisgarh', 'Vegetarian', 'Dessert', 'Fried rice and yogurt dumplings soaked in sugar syrup'),
    ('Bafauri', 'Chhattisgarh', 'Vegetarian', 'Snack', 'Steamed chana dal fritters with onion, ginger and green chilli'),
    ('Aloo ke Gutke', 'Uttarakhand', 'Vegetarian', 'Snack', 'Boiled potatoes fried with cumin, red chilli and coriander'),
    ('Jadoh', 'Meghalaya', 'Non-Vegetarian', 'Lunch', 'Red rice cooked with pork, ginger, onion and turmeric'),
    ('Eromba', 'Manipur', 'Non-Vegetarian', 'Lunch', 'Boiled vegetables mashed with fermented fish and chilli'),
    ('Masor Tenga', 'Assam', 'Non-Vegetarian', 'Lunch', 'Sour fish curry with tomato, lemon and mustard oil'),
    ('Smoked Pork with Bamboo Shoot', 'Nagaland', 'Non-Vegetarian', 'Dinner', 'Smoked pork cooked with fermented bamboo shoot, chilli and ginger'),
    ('Thukpa', 'Sikkim', 'Non-Vegetarian', 'Dinner', 'Noodle soup with chicken, vegetables, garlic and ginger'),
    ('Momos', 'Sikkim', 'Non-Vegetarian', 'Snack', 'Steamed dumplings filled with chicken, onion, garlic and ginger'),
    ('Bai', 'Mizoram', 'Vegetarian', 'Lunch', 'Boiled vegetables with bamboo shoot, spinach and rice'),
    ('Chakhwi', 'Tripura', 'Vegetarian', 'Lunch', 'Vegetable stew with bamboo shoot, papaya and baking soda'),
    ('Aloo Pitika', 'Assam', 'Vegetarian', 'Lunch', 'Mashed boiled potato with mustard oil, onion and green chilli'),
    ('Til Pitha', 'Assam', 'Vegetarian', 'Dessert', 'Rice flour rolls stuffed with sesame and jaggery, roasted on a pan'),
    ('Singju', 'Manipur', 'Vegetarian', 'Snack', 'Salad of cabbage, lotus stem and chilli with roasted gram flour'),
]

def supplement_records():
    return [{'recipe_name': n, 'description': d, 'veg_nonveg': v, 'dish_type': c, 'state': s,
             'region_section': None, 'image_url': None, 'recipe_source_url': None,
             'source': 'Manual supplement'} for n, s, v, c, d in SUPPLEMENT]

### Run the scrapers and save the semi-structured JSON
The GI food items are stored twice in the JSON: as a lookup list (`gi_register`) and as extra recipe records (`source = 'Wikipedia: GI register'`). Raw data is cached in `indian_recipes_raw.json`; delete it to re-scrape.

In [ ]:
if os.path.exists(RAW_JSON):
    print(f'Loading cached {RAW_JSON} (delete it to scrape again)')
else:
    dishes = scrape_dishes()
    print('Source 1 - dishes scraped:', len(dishes))
    print('  per table section:', Counter(d['region_section'] for d in dishes))
    time.sleep(1)
    gi_foods = scrape_gi_foods()
    print('Source 2 - GI food items scraped:', len(gi_foods))
    gi_records = [{'recipe_name': g['gi_name'], 'description': f"GI-tagged food stuff from {g['state']}",
                   'veg_nonveg': None, 'dish_type': None, 'state': g['state'], 'region_section': None,
                   'image_url': None, 'recipe_source_url': GI_URL, 'source': 'Wikipedia: GI register'}
                  for g in gi_foods]
    supplement = supplement_records()
    print('Source 3 - manual supplement:', len(supplement))
    raw = {'recipes': dishes + gi_records + supplement, 'gi_register': gi_foods}
    with open(RAW_JSON, 'w', encoding='utf-8') as f:
        json.dump(raw, f, indent=2, ensure_ascii=False)
    print('Total raw recipe records:', len(raw['recipes']))

---
## (b) JSON → DataFrame
The JSON has two parts: `recipes` (the records) and `gi_register` (lookup table). Each becomes a DataFrame.

In [ ]:
with open(RAW_JSON, encoding='utf-8') as f:
    raw = json.load(f)

print(json.dumps(raw['recipes'][0], indent=2, ensure_ascii=False))
recipes = pd.DataFrame(raw['recipes'])
gi = pd.DataFrame(raw['gi_register'])
print('\nRecipes:', recipes.shape, '| GI register (food stuff):', gi.shape)
print(recipes['source'].value_counts())
recipes.head()

---
## (c) Data wrangling
### c1. Standardising names and removing duplicates across sources

The same dish can appear in several regional tables, in the GI register and in our supplement, with small spelling differences ("Momos" vs "Momo", "Hyderabadi Biryani (Telangana)"). We build a **normalised key**: lower case, remove text in brackets, keep only letters, drop a plural "s". Records with the same key are duplicates; we keep the richest record (Wikipedia dish table first, then supplement, then GI register). In the next step, GI records that are the same food under a different name (e.g. 'Banglar Rasogolla' = 'Rasgulla') are also removed.

In [ ]:
def name_key(name):
    s = re.sub(r'\(.*?\)', ' ', str(name).lower())
    s = re.sub(r'[^a-z ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return re.sub(r's$', '', s) if len(s) > 4 else s

recipes['recipe_name'] = recipes['recipe_name'].str.replace(r'\s+', ' ', regex=True).str.strip()
recipes = recipes[recipes['recipe_name'].str.len() > 1]
recipes['name_key'] = recipes['recipe_name'].apply(name_key)

priority = {'Wikipedia: List of Indian dishes': 0, 'Manual supplement': 1, 'Wikipedia: GI register': 2}
recipes['priority'] = recipes['source'].map(priority)
recipes = recipes.sort_values('priority')

dup_count = recipes.duplicated('name_key').sum()
print('Duplicate dishes across/within sources:', dup_count)
print(recipes[recipes.duplicated('name_key', keep=False)].sort_values('name_key')[['recipe_name', 'source', 'region_section']].head(12))

df = recipes.drop_duplicates('name_key', keep='first').drop(columns='priority').reset_index(drop=True)
print('\nUnique dishes:', len(df))

### c2. GI tag status
A dish is GI-tagged if its normalised name appears inside a GI product name (or the other way round), e.g. "Bikaneri Bhujia" ↔ "Bikaneri Bhujia", "Haleem" ↔ "Hyderabad Haleem". A short alias table handles common spelling variants. Matches also give us the dish's **state**.

In [ ]:
ALIASES = {'rasgulla': 'rasogolla', 'rasagola': 'rasogolla', 'tirupati laddu': 'tirupathi laddu',
           'hyderabadi haleem': 'hyderabad haleem', 'mysore pak': 'mysore pak', 'petha': 'agra petha'}
gi['key'] = gi['gi_name'].apply(name_key)

def gi_match(key):
    """Return (state, gi_key) of the matching GI food item, or (None, None)."""
    key = ALIASES.get(key, key)
    if len(key) < 4:
        return None, None
    for _, g in gi.iterrows():
        if key == g['key'] or (len(key) >= 5 and (re.search(rf'\b{re.escape(key)}\b', g['key'])
                                                  or re.search(rf'\b{re.escape(g["key"])}\b', key))):
            return g['state'], g['key']
    return None, None

is_gi_record = df['source'] == 'Wikipedia: GI register'
matches = df.loc[~is_gi_record, 'name_key'].apply(gi_match)
df['gi_state'] = matches.str[0]
matched_gi_keys = set(matches.str[1].dropna())

# A GI record that already matched a dish (e.g. 'Banglar Rasogolla' = 'Rasgulla') is the same food: drop it
same_food = is_gi_record & df['name_key'].isin(matched_gi_keys)
print('GI register records that duplicate an existing dish (dropped):', df.loc[same_food, 'recipe_name'].tolist())
df = df[~same_food].reset_index(drop=True)
df.loc[df['source'] == 'Wikipedia: GI register', 'gi_state'] = df['state']

df['gi_tag'] = np.where(df['gi_state'].notna(), 'Yes', 'No')
print(df['gi_tag'].value_counts())
df.loc[df.gi_tag == 'Yes', ['recipe_name', 'gi_state', 'source']].head(10)

### c3. State and region: extraction and standardisation

1. **State:** use the state given by the source (supplement, GI register), else look for a state name or a demonym in the name/description ("from Assam", "Bengali", "Hyderabadi", "Goan", "Chettinad"...).
2. **Standardise state spellings** ("Orissa" → "Odisha", "Pondicherry" → "Puducherry", "J&K" → "Jammu and Kashmir").
3. **Region:** from the state via the Zonal Council mapping; if the state is unknown, fall back to the Wikipedia table section the dish was listed under.

In [ ]:
STATE_TO_REGION = {
    # North
    'Punjab': 'North', 'Haryana': 'North', 'Himachal Pradesh': 'North', 'Jammu and Kashmir': 'North',
    'Ladakh': 'North', 'Delhi': 'North', 'Chandigarh': 'North', 'Rajasthan': 'North',
    # Central
    'Uttar Pradesh': 'Central', 'Uttarakhand': 'Central', 'Madhya Pradesh': 'Central', 'Chhattisgarh': 'Central',
    # East
    'West Bengal': 'East', 'Odisha': 'East', 'Bihar': 'East', 'Jharkhand': 'East',
    # West
    'Gujarat': 'West', 'Maharashtra': 'West', 'Goa': 'West',
    # South
    'Kerala': 'South', 'Tamil Nadu': 'South', 'Karnataka': 'South', 'Andhra Pradesh': 'South',
    'Telangana': 'South', 'Puducherry': 'South',
    # North-East
    'Assam': 'North-East', 'Meghalaya': 'North-East', 'Manipur': 'North-East', 'Mizoram': 'North-East',
    'Nagaland': 'North-East', 'Tripura': 'North-East', 'Arunachal Pradesh': 'North-East', 'Sikkim': 'North-East',
}
STATE_SPELLINGS = {'orissa': 'Odisha', 'pondicherry': 'Puducherry', 'j&k': 'Jammu and Kashmir',
                   'jammu & kashmir': 'Jammu and Kashmir', 'kashmir': 'Jammu and Kashmir', 'bengal': 'West Bengal',
                   'uttaranchal': 'Uttarakhand', 'nct of delhi': 'Delhi', 'new delhi': 'Delhi'}
DEMONYMS = {
    'punjabi': 'Punjab', 'kashmiri': 'Jammu and Kashmir', 'himachali': 'Himachal Pradesh', 'rajasthani': 'Rajasthan',
    'marwari': 'Rajasthan', 'bikaneri': 'Rajasthan', 'lucknowi': 'Uttar Pradesh', 'awadhi': 'Uttar Pradesh',
    'banarasi': 'Uttar Pradesh', 'mathura': 'Uttar Pradesh', 'agra': 'Uttar Pradesh', 'kumaoni': 'Uttarakhand',
    'garhwali': 'Uttarakhand', 'indori': 'Madhya Pradesh', 'bhopali': 'Madhya Pradesh', 'bengali': 'West Bengal',
    'kolkata': 'West Bengal', 'odia': 'Odisha', 'oriya': 'Odisha', 'bihari': 'Bihar', 'gujarati': 'Gujarat',
    'surati': 'Gujarat', 'maharashtrian': 'Maharashtra', 'marathi': 'Maharashtra', 'mumbai': 'Maharashtra',
    'kolhapuri': 'Maharashtra', 'malvani': 'Maharashtra', 'puneri': 'Maharashtra', 'goan': 'Goa',
    'kerala': 'Kerala', 'malabar': 'Kerala', 'tamil': 'Tamil Nadu', 'chettinad': 'Tamil Nadu',
    'madurai': 'Tamil Nadu', 'udupi': 'Karnataka', 'mangalorean': 'Karnataka', 'mysore': 'Karnataka',
    'kannada': 'Karnataka', 'coorg': 'Karnataka', 'kodava': 'Karnataka', 'andhra': 'Andhra Pradesh',
    'hyderabadi': 'Telangana', 'hyderabad': 'Telangana', 'assamese': 'Assam', 'manipuri': 'Manipur',
    'naga': 'Nagaland', 'mizo': 'Mizoram', 'khasi': 'Meghalaya', 'sikkimese': 'Sikkim', 'tripuri': 'Tripura',
}

def standardise_state(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    s = str(s).strip()
    low = s.lower()
    if low in STATE_SPELLINGS:
        return STATE_SPELLINGS[low]
    for state in STATE_TO_REGION:
        if state.lower() in low:
            return state
    return None


def find_state(row):
    for given in (row.get('state'), row.get('gi_state')):
        st = standardise_state(given)
        if st:
            return st
    text = f"{row['recipe_name']} {row['description']}".lower()
    for state in sorted(STATE_TO_REGION, key=len, reverse=True):      # longest first: 'West Bengal' before 'Bengal'
        if state.lower() in text:
            return state
    for word, state in {**STATE_SPELLINGS, **DEMONYMS}.items():
        if re.search(rf'\b{re.escape(word)}\b', text):
            return state
    return None

df['state'] = df.apply(find_state, axis=1)
df['region'] = df['state'].map(STATE_TO_REGION).fillna(df['region_section'])
print('Dishes with a known state:', df['state'].notna().sum(), 'of', len(df))
print('Missing region:', df['region'].isnull().sum())
df['region'].value_counts(dropna=False)

### c4. Vegetarian / non-vegetarian: standardise and fill
Raw values vary ("Vegetarian", "Non-Vegetarian", "Non-vegetarian", "Veg", blank). We standardise to two labels. If missing, we infer from ingredients: any meat, fish or egg word → Non-Vegetarian, else Vegetarian.

In [ ]:
NONVEG_WORDS = ['chicken', 'mutton', 'lamb', 'goat', 'meat', 'fish', 'prawn', 'shrimp', 'crab', 'egg',
                'pork', 'beef', 'keema', 'gosht', 'murgh', 'machh', 'machher', 'macher', 'mangsho', 'duck']

def veg_label(row):
    v = str(row['veg_nonveg']).lower()
    if 'non' in v:
        return 'Non-Vegetarian'
    if 'veg' in v:
        return 'Vegetarian'
    text = f"{row['recipe_name']} {row['description']}".lower()
    return 'Non-Vegetarian' if any(re.search(rf'\b{w}', text) for w in NONVEG_WORDS) else 'Vegetarian'

print('Raw values:', df['veg_nonveg'].fillna('<missing>').value_counts().head(8).to_dict())
df['veg_nonveg'] = df.apply(veg_label, axis=1)
df['veg_nonveg'].value_counts()

### c5. Cleaning ingredient lists
The description is free text. We extract **major ingredients** by matching a vocabulary of ~60 ingredients, where several spellings or Hindi names map to one clean name (e.g. *aloo / potato / potatoes* → `potato`, *dahi / curd / yogurt / yoghurt* → `yogurt`). The result is a clean, de-duplicated, comma-separated list.

In [ ]:
INGREDIENTS = {
    'rice': ['rice', 'chawal', 'basmati'], 'wheat flour': ['wheat', 'atta', 'maida', 'flour'],
    'gram flour': ['gram flour', 'besan'], 'lentils': ['lentil', 'dal', 'daal', 'urad', 'moong', 'toor', 'masoor'],
    'chickpeas': ['chickpea', 'chana', 'chole'], 'potato': ['potato', 'aloo', 'alu'], 'onion': ['onion'],
    'tomato': ['tomato'], 'garlic': ['garlic'], 'ginger': ['ginger'], 'chilli': ['chilli', 'chili', 'chile', 'mirchi'],
    'coconut': ['coconut'], 'yogurt': ['yogurt', 'yoghurt', 'curd', 'dahi'], 'milk': ['milk', 'khoya', 'khoa', 'mawa'],
    'ghee': ['ghee'], 'butter': ['butter'], 'cream': ['cream', 'malai'], 'paneer': ['paneer', 'cottage cheese', 'chhena', 'chenna'],
    'sugar': ['sugar', 'syrup'], 'jaggery': ['jaggery', 'gur'], 'cardamom': ['cardamom', 'elaichi'],
    'saffron': ['saffron', 'kesar'], 'turmeric': ['turmeric', 'haldi'], 'cumin': ['cumin', 'jeera'],
    'mustard': ['mustard'], 'coriander': ['coriander', 'cilantro', 'dhania'], 'curry leaves': ['curry leaves', 'curry leaf'],
    'tamarind': ['tamarind', 'imli'], 'garam masala': ['garam masala'], 'spices': ['spice', 'masala'],
    'chicken': ['chicken', 'murgh'], 'mutton': ['mutton', 'goat', 'lamb', 'gosht'], 'fish': ['fish', 'machh', 'macher'],
    'prawn': ['prawn', 'shrimp'], 'egg': ['egg'], 'pork': ['pork'], 'spinach': ['spinach', 'palak', 'saag'],
    'cauliflower': ['cauliflower', 'gobi'], 'peas': ['peas', 'matar'], 'eggplant': ['eggplant', 'brinjal', 'baingan', 'aubergine'],
    'okra': ['okra', 'bhindi', 'ladyfinger'], 'cabbage': ['cabbage'], 'carrot': ['carrot', 'gajar'],
    'pumpkin': ['pumpkin', 'gourd'], 'bamboo shoot': ['bamboo shoot'], 'mushroom': ['mushroom'],
    'semolina': ['semolina', 'rava', 'sooji', 'suji'], 'vermicelli': ['vermicelli', 'sevai', 'noodle'],
    'nuts': ['cashew', 'almond', 'pistachio', 'nuts', 'walnut'], 'raisins': ['raisin', 'kishmish'],
    'sesame': ['sesame', 'til'], 'peanut': ['peanut', 'groundnut'], 'banana': ['banana', 'plantain'],
    'jackfruit': ['jackfruit'], 'mango': ['mango'], 'lemon': ['lemon', 'lime'], 'corn': ['corn', 'maize', 'makki'],
    'beans': ['beans', 'rajma', 'kidney bean'], 'pepper': ['pepper'], 'flattened rice': ['flattened rice', 'poha', 'chivda'],
}

def extract_ingredients(text):
    text = str(text).lower()
    found = [name for name, words in INGREDIENTS.items()
             if any(re.search(rf'\b{re.escape(w)}', text) for w in words)]
    if 'gram flour' in found and 'wheat flour' in found and 'flour' not in text.replace('gram flour', ''):
        found.remove('wheat flour')                  # 'gram flour' alone should not also count as wheat flour
    if 'flattened rice' in found and 'rice' in found and 'rice' not in text.replace('flattened rice', ''):
        found.remove('rice')
    return found

df['ingredient_list'] = (df['recipe_name'] + ' ' + df['description']).apply(extract_ingredients)
df['major_ingredients'] = df['ingredient_list'].apply(lambda l: ', '.join(l) if l else 'Not specified')
df[['recipe_name', 'description', 'major_ingredients']].head(8)

### c6. Cooking method, category and recommended meal time

**Cooking method** — first match wins: Fried (fried, fry, deep-fried) → Steamed → Grilled/Roasted (roast, grill, tandoor, barbecue, skewer) → Baked → Fermented → Boiled → Sweet/Syrup (sugar syrup, candy, sweet) → Curry/Simmered (curry, gravy, stew, simmer, soup, cooked) → Raw/Mixed (salad, chutney, mixed).

**Category** — Wikipedia's "Dish Type" column if present, else keyword rules: Dessert (sweet words) → Breakfast (dosa, idli, poha, paratha...) → Snack (pakora, samosa, chaat...) → Lunch (rice, dal, sambar, thali...) → Dinner (everything else: curries, biryanis, meat dishes).

**Recommended meal time:** Breakfast → Morning, Lunch → Afternoon, Snack → Evening, Dinner → Night, Dessert → After meals.

In [ ]:
METHODS = [
    ('Fried', ['deep fried', 'deep-fried', 'fried', 'fry', 'fritter', 'pakora', 'pakoda', 'bhaji', 'samosa']),
    ('Steamed', ['steamed', 'steam', 'idli', 'dhokla', 'momo']),
    ('Grilled/Roasted', ['roast', 'grill', 'tandoor', 'barbecue', 'skewer', 'kebab', 'kabab', 'tikka', 'griddle']),
    ('Baked', ['baked', 'bake', 'oven']),
    ('Fermented', ['fermented', 'ferment', 'batter']),
    ('Boiled', ['boiled', 'boil']),
    ('Sweet/Syrup', ['syrup', 'candy', 'sweet', 'dessert', 'porridge', 'pudding']),
    ('Curry/Simmered', ['curry', 'gravy', 'stew', 'simmer', 'soup', 'cooked', 'masala', 'korma', 'sauce', 'dal', 'sambar']),
    ('Raw/Mixed', ['salad', 'chutney', 'raita', 'mixed', 'mashed', 'pickle']),
]
CATEGORIES = [
    ('Dessert', ['sweet', 'dessert', 'kheer', 'halwa', 'laddu', 'ladoo', 'laddoo', 'barfi', 'burfi', 'pedha', 'peda',
                 'jalebi', 'payasam', 'rasgulla', 'rasogolla', 'sandesh', 'gulab', 'mithai', 'kulfi', 'malpua', 'rabri',
                 'shrikhand', 'modak', 'petha', 'mysore pak', 'chikki', 'sondesh', 'mishti', 'syrup', 'pudding', 'jaggery']),
    ('Breakfast', ['breakfast', 'dosa', 'idli', 'upma', 'poha', 'paratha', 'uttapam', 'appam', 'puttu', 'pongal',
                   'idiyappam', 'thepla', 'chila', 'chilla', 'pesarattu', 'sabudana', 'puri', 'poori', 'pancake']),
    ('Snack', ['snack', 'pakora', 'pakoda', 'samosa', 'chaat', 'vada', 'bhaji', 'bhajia', 'kachori', 'dhokla', 'bhujia',
               'namkeen', 'chips', 'fritter', 'tikki', 'momo', 'pani puri', 'golgappa', 'sev', 'khakhra', 'murukku',
               'street food', 'mixture', 'salad', 'pickle', 'chutney']),
    ('Lunch', ['rice', 'dal', 'daal', 'sambar', 'rasam', 'thali', 'khichdi', 'pulao', 'curd rice', 'lentil']),
]
MEAL_TIME = {'Breakfast': 'Morning', 'Lunch': 'Afternoon', 'Snack': 'Evening', 'Dinner': 'Night', 'Dessert': 'After meals'}

def first_match(text, rules, default):
    for label, words in rules:
        if any(re.search(rf'\b{re.escape(w)}', text) for w in words):
            return label
    return default

def category(row):
    hint = str(row['dish_type']).lower()
    for label in ['Dessert', 'Breakfast', 'Snack', 'Lunch', 'Dinner']:
        if label.lower() in hint:
            return label
    if 'main' in hint:
        return 'Lunch' if 'rice' in str(row['description']).lower() else 'Dinner'
    if 'sweet' in hint:
        return 'Dessert'
    return first_match(f"{row['recipe_name']} {row['description']}".lower(), CATEGORIES, 'Dinner')

text = (df['recipe_name'] + ' ' + df['description']).str.lower()
df['cooking_method'] = text.apply(lambda t: first_match(t, METHODS, 'Other'))
df['category'] = df.apply(category, axis=1)
df['recommended_meal_time'] = df['category'].map(MEAL_TIME)
pd.crosstab(df['category'], df['cooking_method'])

### c7. Missing values: overview and handling

In [ ]:
df['prep_time_min'] = np.nan          # not published by our sources
df['calories_kcal'] = np.nan          # not published by our sources
missing = df.isnull().sum()
pd.DataFrame({'missing': missing, '%': (missing / len(df) * 100).round(1)}).query('missing > 0')

| Column | Handling | Reason |
|---|---|---|
| `prep_time_min`, `calories_kcal` | **drop** | 100% missing: our sources do not publish them; imputing would invent data. We engineer a *complexity* feature instead. |
| `state` | fill `'Unknown'` | Many Wikipedia descriptions do not name a state; the region is still known from the table section. |
| `region` | fill `'Unknown'` (none expected) | Every dish has a state-based or table-based region. |
| `image_url` | fill `'Not available'` | GI and supplement records have no image. |
| `recipe_source_url` | fill `'Not available'` | Supplement records were not scraped. |
| `dish_type`, `region_section`, `gi_state` | drop after use | Helper columns, already used to build category, region and GI tag. |

In [ ]:
df = df.drop(columns=['prep_time_min', 'calories_kcal', 'dish_type', 'region_section', 'gi_state'])
df['state'] = df['state'].fillna('Unknown')
df['region'] = df['region'].fillna('Unknown')
df['image_url'] = df['image_url'].fillna('Not available')
df['recipe_source_url'] = df['recipe_source_url'].fillna('Not available')
print('Missing values left:', df.isnull().sum().sum())

### c8. Converting to suitable types
Repeated text labels with a small fixed set of values become pandas **`category`** columns (less memory, and plots keep a sensible order). Region and category get an explicit order.

In [ ]:
df['region'] = pd.Categorical(df['region'], categories=['North', 'Central', 'East', 'West', 'South', 'North-East', 'Unknown'])
df['category'] = pd.Categorical(df['category'], categories=['Breakfast', 'Lunch', 'Snack', 'Dinner', 'Dessert'])
for c in ['veg_nonveg', 'gi_tag', 'cooking_method', 'recommended_meal_time', 'source']:
    df[c] = df[c].astype('category')
df.dtypes

---
## (d) Feature engineering (6 new features) and normalization

| Feature | Definition |
|---|---|
| `ingredient_count` | number of major ingredients extracted |
| `spice_level` | Mild / Medium / Hot from the number of "heat" words (chilli, pepper, garam masala, spicy, hot, vindaloo, Chettinad, Kolhapuri...) |
| `complexity` | Easy / Medium / Hard from a score = ingredient count + method effort (fried, fermented, baked +2; curry, grilled +1) |
| `regional_cuisine` | cuisine name from the state (Punjabi, Bengali, Gujarati...) or "<Region> Indian" |
| `health_score` | 0–10 heuristic: +1 per vegetable/lentil/fish/fruit ingredient (max +3), +1.5 if steamed/boiled/grilled, −2 if fried, −2 for desserts, −1 for ghee/butter/cream, then clipped to [0, 10] |
| `description_length` | number of words in the description |

The health score is a **simple rule-based estimate** for comparison only, not a nutritional measurement.

In [ ]:
HEAT_WORDS = ['chilli', 'chili', 'pepper', 'garam masala', 'spicy', 'hot', 'vindaloo', 'chettinad', 'kolhapuri', 'mirchi', 'masala']
HEALTHY = {'lentils', 'chickpeas', 'spinach', 'cauliflower', 'peas', 'eggplant', 'okra', 'cabbage', 'carrot',
           'pumpkin', 'bamboo shoot', 'mushroom', 'tomato', 'fish', 'beans', 'banana', 'mango', 'lemon', 'corn', 'coconut'}
CUISINE = {'Punjab': 'Punjabi', 'Jammu and Kashmir': 'Kashmiri', 'Rajasthan': 'Rajasthani', 'Uttar Pradesh': 'Awadhi/UP',
           'Madhya Pradesh': 'Malwa/MP', 'Chhattisgarh': 'Chhattisgarhi', 'West Bengal': 'Bengali', 'Odisha': 'Odia',
           'Bihar': 'Bihari', 'Gujarat': 'Gujarati', 'Maharashtra': 'Maharashtrian', 'Goa': 'Goan', 'Kerala': 'Kerala',
           'Tamil Nadu': 'Tamil', 'Karnataka': 'Karnataka', 'Andhra Pradesh': 'Andhra', 'Telangana': 'Hyderabadi/Telangana',
           'Assam': 'Assamese', 'Manipur': 'Manipuri', 'Meghalaya': 'Khasi/Meghalaya', 'Nagaland': 'Naga',
           'Sikkim': 'Sikkimese', 'Mizoram': 'Mizo', 'Tripura': 'Tripuri', 'Uttarakhand': 'Kumaoni/Garhwali'}
METHOD_EFFORT = {'Fried': 2, 'Fermented': 2, 'Baked': 2, 'Curry/Simmered': 1, 'Grilled/Roasted': 1}

df['ingredient_count'] = df['ingredient_list'].str.len()
heat = text.loc[df.index].apply(lambda t: sum(bool(re.search(rf'\b{w}', t)) for w in HEAT_WORDS))
df['spice_level'] = pd.cut(heat, bins=[-1, 0, 1, 100], labels=['Mild', 'Medium', 'Hot'])
df.loc[df['category'] == 'Dessert', 'spice_level'] = 'Mild'

score = df['ingredient_count'] + df['cooking_method'].astype(str).map(METHOD_EFFORT).fillna(0)
df['complexity'] = pd.cut(score, bins=[-1, 2, 4, 100], labels=['Easy', 'Medium', 'Hard'])

df['regional_cuisine'] = df['state'].map(CUISINE).fillna(df['region'].astype(str) + ' Indian')

def health(row):
    s = 5.0
    s += min(3, sum(i in HEALTHY for i in row['ingredient_list']))
    if row['cooking_method'] in ('Steamed', 'Boiled', 'Grilled/Roasted'): s += 1.5
    if row['cooking_method'] == 'Fried': s -= 2
    if row['category'] == 'Dessert': s -= 2
    if any(i in ('ghee', 'butter', 'cream') for i in row['ingredient_list']): s -= 1
    return float(np.clip(s, 0, 10))

df['health_score'] = df.apply(health, axis=1)
df['description_length'] = df['description'].str.split().str.len()

df[['recipe_name', 'region', 'category', 'cooking_method', 'ingredient_count', 'spice_level',
    'complexity', 'regional_cuisine', 'health_score']].head(10)

### Normalization

**Technique: Min-Max scaling** for `ingredient_count`, `description_length` and `health_score`.

**Justification:** all three are **bounded counts/scores** (ingredients 0–~12, health 0–10) with no extreme outliers, and they live on different ranges. Min-Max maps each to [0, 1] while keeping the shape of the distribution, so they can be compared directly or combined into a single index (for example a "healthy and simple" ranking). Z-score scaling would be preferable only if we had strong outliers or needed zero-centred inputs for a model.

In [ ]:
num = ['ingredient_count', 'description_length', 'health_score']
df[[c + '_norm' for c in num]] = MinMaxScaler().fit_transform(df[num])
df['healthy_simple_index'] = ((df['health_score_norm'] + (1 - df['ingredient_count_norm'])) / 2).round(3)
df[num + [c + '_norm' for c in num]].describe().loc[['min', 'max', 'mean']].round(3)

### Final structured dataset

In [ ]:
final_cols = ['recipe_name', 'state', 'region', 'regional_cuisine', 'category', 'recommended_meal_time',
              'veg_nonveg', 'major_ingredients', 'ingredient_count', 'cooking_method', 'spice_level', 'complexity',
              'gi_tag', 'health_score', 'description', 'description_length', 'image_url', 'recipe_source_url',
              'source', 'ingredient_count_norm', 'description_length_norm', 'health_score_norm', 'healthy_simple_index']
final = df[final_cols].sort_values(['region', 'recipe_name']).reset_index(drop=True)
print('Final dataset:', final.shape, '| unique recipes:', final['recipe_name'].nunique())
print('Regions covered:', final['region'].value_counts().to_dict())
final.head()

---
## (e) Exploratory data analysis — answering the analytical questions
### i. Which region has the largest variety of recipes?

In [ ]:
region_counts = final['region'].value_counts()
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
region_counts.plot(kind='bar', ax=ax[0], rot=0, color=sns.color_palette('Set2', len(region_counts)))
ax[0].set(title='1. Number of unique recipes per region', ylabel='Recipes')
for i, v in enumerate(region_counts):
    ax[0].text(i, v + 1, str(v), ha='center')
cuisine_counts = final[~final['regional_cuisine'].str.endswith(' Indian')]['regional_cuisine'].value_counts().head(12)
cuisine_counts.sort_values().plot(kind='barh', ax=ax[1], color='tab:blue')
ax[1].set(title='1b. Recipes per regional cuisine (state known)', xlabel='Recipes')
plt.tight_layout()
plt.show()
print('Answer: the region with the largest variety is', region_counts.idxmax(), f'({region_counts.max()} recipes).')

### ii. Which ingredients are most frequently used across India?

In [ ]:
ingredient_freq = df['ingredient_list'].explode().value_counts()
ingredient_freq = ingredient_freq[ingredient_freq.index != 'spices']
plt.figure(figsize=(10, 7))
ingredient_freq.head(20).sort_values().plot(kind='barh', color='tab:orange')
plt.title('2. Top 20 most frequently mentioned ingredients')
plt.xlabel('Number of recipes')
plt.show()
print('Answer: most frequent ingredients:', ingredient_freq.head(8).to_dict())

### iii. What are the most common cooking methods?

In [ ]:
method_counts = final['cooking_method'].value_counts()
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
method_counts.plot(kind='bar', ax=ax[0], rot=25, color='tab:green')
ax[0].set(title='3. Cooking methods', ylabel='Recipes')
pd.crosstab(final['region'], final['cooking_method'], normalize='index').mul(100).plot(
    kind='bar', stacked=True, ax=ax[1], rot=0, colormap='tab20')
ax[1].set(title='3b. Cooking method mix per region (%)', ylabel='% of recipes')
ax[1].legend(bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()
print('Answer: most common cooking methods:', method_counts.head(4).to_dict())

### iv. Which states possess the highest number of GI-tagged foods?
Answered from the GI register itself (all GI food stuff), which is the authoritative list.

In [ ]:
gi['state_std'] = gi['state'].apply(standardise_state).fillna(gi['state'])
gi_states = gi['state_std'].value_counts()
plt.figure(figsize=(10, 6))
gi_states.head(15).sort_values().plot(kind='barh', color='tab:purple')
plt.title('4. States with the most GI-tagged food products')
plt.xlabel('GI-tagged food items')
plt.show()
print('Answer: top states by GI-tagged foods:', gi_states.head(5).to_dict())
print('GI-tagged dishes in our final dataset:', (final['gi_tag'] == 'Yes').sum())

### v. What proportion of dishes are vegetarian vs non-vegetarian?

In [ ]:
veg_share = final['veg_nonveg'].value_counts()
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].pie(veg_share, labels=veg_share.index, autopct='%1.1f%%', colors=['#66bb6a', '#ef5350'], startangle=90)
ax[0].set_title('5. Vegetarian vs non-vegetarian (all India)')
pd.crosstab(final['region'], final['veg_nonveg'], normalize='index').mul(100).plot(
    kind='bar', stacked=True, ax=ax[1], rot=0, color=['#ef5350', '#66bb6a'])
ax[1].set(title='5b. Veg / non-veg share by region (%)', ylabel='%')
plt.tight_layout()
plt.show()
print('Answer:', (veg_share / veg_share.sum() * 100).round(1).to_dict())

### vi. Which meal category dominates each region?

In [ ]:
region_cat = pd.crosstab(final['region'], final['category'])
plt.figure(figsize=(10, 5.5))
sns.heatmap(region_cat, annot=True, fmt='d', cmap='YlOrBr')
plt.title('6. Recipes by region and meal category')
plt.show()
dominant = region_cat.idxmax(axis=1)
print('Answer — dominant meal category per region:')
print(pd.DataFrame({'dominant category': dominant, 'recipes': region_cat.max(axis=1)}))

### Further visualizations

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(x='category', y='health_score', data=final, ax=ax[0], palette='Set3')
ax[0].set_title('7. Health score by meal category')
pd.crosstab(final['region'], final['spice_level'], normalize='index').mul(100)[['Mild', 'Medium', 'Hot']].plot(
    kind='bar', stacked=True, ax=ax[1], rot=0, color=['#fff3b0', '#f4a259', '#bc4b51'])
ax[1].set(title='8. Spice level mix by region (%)', ylabel='%')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
sns.histplot(final['ingredient_count'], discrete=True, ax=ax[0], color='tab:cyan')
ax[0].set(title='9. Number of major ingredients per recipe', xlabel='Ingredients')
sns.heatmap(final[['ingredient_count', 'health_score', 'description_length', 'healthy_simple_index']].corr(),
            annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax[1])
ax[1].set_title('10. Correlation among numeric features')
plt.tight_layout()
plt.show()

---
## Save the dataset

In [ ]:
final.to_csv('indian_recipes_clean.csv', index=False, encoding='utf-8')
print('Saved indian_recipes_clean.csv', final.shape)

## Analytical report — major findings

The **numbers** come from the answers printed under questions i–vi, so fill them in from your run. Points 3–7 also say what to look for, based on what is commonly known about Indian cuisine; keep a point only if your output supports it.

1. **Coverage:** the final dataset has 200+ unique dishes from all six regions (the region counts are printed above the EDA section). They come from three sources, de-duplicated with a normalised name key and a GI-name match. Wikipedia's tables are richest for the West, South and North; Central and North-East coverage relies partly on the labelled manual supplement.
2. **Variety by region (Q i):** state the region with the most recipes and its count. This partly reflects how well each region's cuisine is documented online, not only how varied it is.
3. **Shared pantry (Q ii):** list the top 5 ingredients from the printout. Expect staples such as rice, lentils, onion, chilli, coconut, yogurt and the aromatics (ginger, garlic) to dominate. Use the region-wise plots to check whether coconut is concentrated in South and coastal West recipes.
4. **Cooking styles (Q iii):** name the top 2–3 methods with their counts. Then use plot 3b to compare regions, e.g. whether steaming is more common in the South (idli) and North-East (momos), and grilling/tandoor in the North.
5. **GI tags (Q iv):** name the top states by GI-tagged foods with their counts. GI food products are mostly traditional sweets and snacks with a strong link to a place (e.g. Bikaneri Bhujia, Dharwad Pedha).
6. **Vegetarian share (Q v):** give the overall veg/non-veg percentages and the region with the highest non-vegetarian share (plot 5b).
7. **Meal patterns (Q vi):** give the dominant category for each region from the printed table, and point out any region where breakfast or dessert stands out.

**Limitations:** category, cooking method, spice level and health score are **rule-based estimates** from short descriptions; preparation time and nutrition were not available from these sources. A production dataset would add a recipe site with structured cooking times and nutrition facts.